# 2.1 航空機観測画像, 2.3 撮影カメラのGPSデータ作成

動画の撮影開始日時：
2021-09-29 14:54:50

In [3]:
import os
import ffmpeg
import cv2
import sys

mv_name = 'GP030003_eye3_trim.MP4'
# 動画をトリミングする関数
def ffmpeg_trim_mv(video_path, trim_start, output_name, trim_end='', target_fps=None):
    print(f'==== running: ffmpeg, start {trim_start}, end {trim_end}')
    probe = ffmpeg.probe(video_path)
    duration = int(float(probe['format']['duration']))
    input_duration = int(trim_start.split(':')[0]) * 60 * 60 + int(trim_start.split(':')[1]) * 60 + int(trim_start.split(':')[2])
    #file_path = os.path.dirname(video_path)
    file_path = os.getcwd()

    if trim_end == '':
        end_duration = duration - input_duration
    else:
        end_duration = int(trim_end.split(':')[0]) * 60 * 60 + int(trim_end.split(':')[1]) * 60 + int(trim_end.split(':')[2]) - input_duration

    fps_filter = f'fps={target_fps}' if target_fps else 'copy'
    err = os.system(f'yes | ffmpeg -ss {trim_start} -i {video_path} -t {end_duration} -vf "{fps_filter},scale=trunc(iw/2)*2:trunc(ih/2)*2" -c:v libx264 -c:a aac -strict experimental {os.path.join(file_path, output_name)}')

    if err:
        print("FATAL: command failed")
        #sys.exit(err)

## トリミング 日時 06:03:10 (14:54:50 + 00:08:20) ~ 06:11:55 (14:54:50 + 00:17:05)
# video_path: 航空機観測動画, trim_start: 動画の何秒から使うか, output_name: 出力動画の名前, trim_end: 動画の何秒までを使うか, '' なら動画の最後を指定, target_fps: 出力動画のfpsを指定できる
ffmpeg_trim_mv('/home/rc/TyMindulle_MV/GP030003_eye3.MP4', '00:08:20', mv_name, '00:17:05', target_fps=2)

==== running: ffmpeg, start 00:08:20, end 00:17:05


ffmpeg version 4.2.7-0ubuntu0.1 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 9 (Ubuntu 9.4.0-1ubuntu1~20.04.1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-avresample --disable-filter=resample --enable-avisynth --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librsvg --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --e

In [2]:
## 指定時間のフレームを取り出す処理
# import ffmpeg

# tpoint = 10
# # 取り出したい静止画の時間（tpoint = 10 なら動画の10秒時点の静止画が取れる）

# imagename = '/home/rc/TyMindulle_MV/GP030003_eye3_frame.jpg'
# #取り出したい静止画のファイル名を指定

# stream = ffmpeg.input('/home/rc/TyMindulle_MV/GP030003_eye3_trim.MP4')
# # sample.mp4に切り取りたい動画を入れる

# stream = ffmpeg.output(stream, imagename, ss=tpoint, vframes=1, f='image2')
# ffmpeg.run(stream)


In [2]:
# 動画をクロップする関数
def ffmpeg_crop_mv(video_path, upper_left_x, upper_left_y, width, height, output_name):
    input_file = ffmpeg.input(video_path)
    stream = input_file.crop(x=upper_left_x, y=upper_left_y, width=width, height=height)
    stream = ffmpeg.output(stream, output_name)

    try:
        ffmpeg.run(stream, overwrite_output=True, capture_stderr=True)
    except ffmpeg.Error as e:
        print("FFmpeg error:")
        print(e.stderr.decode())  # 例外処理　エラーの詳細を表示する

## クロップ
# 画像は左上が座標の中心で、upper_left_x, upper_left_yでクロップの開始地点を指定, 開始地点からｘ座標に1574 pixel,y座標に900 pixcelの範囲をクロップ
ffmpeg_crop_mv(
    video_path=os.path.join(os.getcwd(),mv_name),
    upper_left_x=0,
    upper_left_y=0,
    width=1575,
    height=900,
    output_name=os.path.join(os.getcwd(), "GP030003_eye3_crop.MP4")
)

In [4]:
video_file1 = os.path.join(os.getcwd(), "GP030003_eye3_crop.MP4")
video1 = cv2.VideoCapture(video_file1)
print(type(video1))
print(video1.isOpened())

<class 'cv2.VideoCapture'>
True


In [7]:
print("幅: ", video1.get(cv2.CAP_PROP_FRAME_WIDTH))

print("高さ: ", video1.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS: ", video1.get(cv2.CAP_PROP_FPS))

print("フレーム数", video1.get(cv2.CAP_PROP_FRAME_COUNT))
frame_count = int(video1.get(cv2.CAP_PROP_FRAME_COUNT))
#動画の秒数
print("動画の秒数", video1.get(cv2.CAP_PROP_FRAME_COUNT) / video1.get(cv2.CAP_PROP_FPS))

幅:  1574.0
高さ:  900.0
FPS:  2.0
フレーム数 1050.0
動画の秒数 525.0


In [6]:
import ffmpeg

## フレーム毎の秒数を計算する関数
def calculate_frame_time_variable_framerate(video_path, frame_number):
    """
    Calculate the time of a specific frame in a variable frame rate video.

    Parameters:
        video_path (str): The path to the video file.
        frame_number (int): The frame number for which to calculate the time.

    Returns:
        float: The time of the specified frame in seconds.
    """
    probe = ffmpeg.probe(video_path, show_entries='frame=best_effort_timestamp_time')
    frame_timestamps = [float(frame['best_effort_timestamp_time']) for frame in probe['frames']]

    # タイムスタンプが欠損している場合は、前のフレームのタイムスタンプを使う
    molecule, denominator = tuple(probe['streams'][0]['r_frame_rate'].split('/'))
    for i in range(1, len(frame_timestamps)):
        if frame_timestamps[i] == frame_timestamps[i - 1]:
            frame_timestamps[i] = frame_timestamps[i - 1] + 1 / float(molecule)/float(denominator)

    # タイムスタンプのリストを元に、指定されたフレームの時刻を計算
    frame_time = frame_timestamps[frame_number]

    return frame_time, frame_timestamps, probe

# テスト用の値を設定します
video_path = video_file1  # 動画ファイルのパス
frame_number = frame_count - 1  # フレーム番号1050

# 指定されたフレームの時刻を計算
frame_time, frame_timestamps, probe = calculate_frame_time_variable_framerate(video_file1, frame_number)
print(f"フレーム {frame_number} の時刻は {frame_time:.2f} 秒です。")

フレーム 1049 の時刻は 524.50 秒です。


In [8]:
from datetime import timedelta
import datetime as dt

# 動画の撮影開始時刻を指定
GP030003_eye3_str_time = '2021-09-29 14:54:50'
datetime_obj1 = dt.datetime.strptime(GP030003_eye3_str_time, '%Y-%m-%d %H:%M:%S')
print(datetime_obj1)

2021-09-29 14:54:50


In [9]:
# trime_time:使用した動画をトリムミングした時間, mm:分, ss:秒
mm,ss = 8,20
trime_time = mm*60 + ss
frame_times = [datetime_obj1 + timedelta(seconds=trime_time) +timedelta(seconds=frame_timestamps[i]) for i in range(len(frame_timestamps))]

In [10]:
# 動画のフレーム数だけ画像名を作成する
frame_n = [str(y + 1).zfill(4) + '.png' for y in range(len(frame_timestamps))]

In [11]:
import numpy as np
import pandas as pd

# 画像毎の撮影日時をデータフレーム型で保存
frame_timestamps = pd.DataFrame(columns =['img_n', 'time'])
frame_timestamps['img_n'] = frame_n
frame_timestamps['time'] = frame_times
frame_timestamps.to_csv(f'{os.getcwd()}/GP030003_eye3_crop_fps2.csv', index = None)

In [12]:
frame_timestamps

,img_n,time
0,0001.png,2021-09-29 15:03:10.000
1,0002.png,2021-09-29 15:03:10.500
2,0003.png,2021-09-29 15:03:11.000
3,0004.png,2021-09-29 15:03:11.500
4,0005.png,2021-09-29 15:03:12.000
...,...,...
1045,1046.png,2021-09-29 15:11:52.500
1046,1047.png,2021-09-29 15:11:53.000
1047,1048.png,2021-09-29 15:11:53.500
1048,1049.png,2021-09-29 15:11:54.000


In [13]:
# 撮影カメラのフライトデータ（GPSデータ）
flightpath = pd.read_csv(os.path.join(os.getcwd(),"flightpath_simple_20210929.csv"))

In [14]:
flightpath['年月日時間'] = 0 # 初期値を０
for b in range(len(flightpath)):
    flightpath.loc[b,'年月日時間'] = dt.datetime(flightpath['年'][b], flightpath['月'][b], flightpath['日'][b], flightpath['時'][b] + 9, flightpath['分'][b], flightpath['秒'][b])

In [15]:
frame_timestamps['time'] = pd.to_datetime(frame_timestamps['time'])
flightpath['年月日時間'] = pd.to_datetime(flightpath['年月日時間'])

In [16]:
## GPSデータは画像の一秒ごとに対応しているため, 0.5秒ごとの緯度、経度、高度データを前後のフレームに対応するデータで線形補間する

data = pd.DataFrame()

for i in range(0, len(flightpath)):
    if i != 0:
        data.loc[i,0] = np.round((flightpath.iloc[i,7] + flightpath.iloc[i - 1,7])/2, 4)
        data.loc[i,1] = np.round((flightpath.iloc[i,8] + flightpath.iloc[i - 1,8])/2, 4)
        data.loc[i,2] = np.round((flightpath.iloc[i,9] + flightpath.iloc[i - 1,9])/2, 4)
        time_diff = flightpath.iloc[i, 14] - flightpath.iloc[i-1, 14]
        data.loc[i, 3] = flightpath.iloc[i-1, 14] + time_diff / 2

In [17]:
flightpath_data = flightpath.copy()

In [18]:
## 上で線形補完したGPSデータを作成したので、それを元のGPSデータに0.5秒毎のデータとして追加する

result_list = []
count = 0
for i, j in zip(flightpath[['経度', '緯度', '高度', '年月日時間']].values, data.values):
    result_list.append(list(i))
    result_list.append(list(j))
    count += 1*2
result_list.append(flightpath[['経度', '緯度', '高度', '年月日時間']].values[-1])
flightpath = pd.DataFrame(data=result_list, columns=['経度', '緯度', '高度', '年月日時間'])

In [19]:
em_list = [0 for c in range(len(frame_timestamps))]

In [20]:
len(em_list)

1050

In [21]:
## 1,050個の０のデータを作成

df = pd.DataFrame(em_list, columns=['img_n'])
df['緯度'] = 0
df['経度'] = 0
df['高度'] = 0

In [22]:
## フライトデータの日時と画像の撮影日時を合わせる
for c in range(len(frame_timestamps)):
    df.loc[c,'緯度'] = flightpath.loc[flightpath['年月日時間'] == frame_timestamps['time'][c]]['緯度'].values[0]
    df.loc[c,'経度'] = flightpath.loc[flightpath['年月日時間'] == frame_timestamps['time'][c]]['経度'].values[0]
    df.loc[c,'高度'] = flightpath.loc[flightpath['年月日時間'] == frame_timestamps['time'][c]]['高度'].values[0]
    df.loc[c,'img_n'] = frame_timestamps['img_n'][c]

In [23]:
df = df.reset_index(drop=True)

In [26]:
# 画像毎のGPSデータをテキストファイルに書き込む
# COLMAPでGeoreferencingするには、以下の形式で保存する必要がある
with open(os.path.join(os.getcwd(), "mvgps.txt"), 'w') as file:
    for _, row in df.iterrows():
        line = f"{row['img_n']} {row['緯度']} {row['経度']} {row['高度']}\n"
        print(line)
        file.write(line)

0001.png 24.131 134.998 14415.52

0002.png 24.131 134.999 14415.52

0003.png 24.131 135.0 14415.52

0004.png 24.1315 135.001 14415.475

0005.png 24.132 135.002 14415.43

0006.png 24.1325 135.0035 14415.3

0007.png 24.133 135.005 14415.17

0008.png 24.133 135.006 14415.005

0009.png 24.133 135.007 14414.84

0010.png 24.1335 135.008 14414.7

0011.png 24.134 135.009 14414.56

0012.png 24.134 135.01 14414.49

0013.png 24.134 135.011 14414.42

0014.png 24.1345 135.012 14414.415

0015.png 24.135 135.013 14414.41

0016.png 24.135 135.014 14414.415

0017.png 24.135 135.015 14414.42

0018.png 24.1355 135.016 14414.38

0019.png 24.136 135.017 14414.34

0020.png 24.1365 135.018 14414.35

0021.png 24.137 135.019 14414.36

0022.png 24.137 135.0205 14414.38

0023.png 24.137 135.022 14414.4

0024.png 24.1375 135.023 14414.44

0025.png 24.138 135.024 14414.48

0026.png 24.138 135.025 14414.545

0027.png 24.138 135.026 14414.61

0028.png 24.1385 135.027 14414.735

0029.png 24.139 135.028 14414.86

0030